In [0]:
from pyspark.sql.functions import *


df_ember_intensity_bronze = spark.read.table("portfolio_energia.bronze.ember_intensity")
df_ember_demand_bronze = spark.read.table("portfolio_energia.bronze.ember_demand")


def run_ember_eda(df, nome_tabela):
    df.groupBy('iso3', 'date').count().filter(col('count') > 1).show(truncate=False)
    #puxando valores duplicatos de acordo com granularidade
    df_nulo = df.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns 
    ])
    display(df_nulo)
    #verifica valores nulos em todas as colunas

    df.filter(col("iso3").isNull()).select("country").distinct().show(truncate=False)
    #verifica quantidade de iso3 nulo

    df_real = df.withColumn("date_formatada", to_date(col("date"))).filter(col("iso3").isNotNull())
    
    df_limites_por_pais = df_real.groupBy("iso3").agg(
        min("date_formatada").alias("data_inicial"),
        max("date_formatada").alias("data_final")
    )

    df_cenario_ideal = df_limites_por_pais.select(
        "iso3",
        explode(sequence(col("data_inicial"), col("data_final"), expr("INTERVAL 1 MONTH"))).alias("data_esperada")
    ) 

    df_datas_faltantes = df_cenario_ideal.join(
        df_real,
        (df_cenario_ideal["iso3"] == df_real["iso3"]) & 
        (df_cenario_ideal["data_esperada"] == df_real["date_formatada"]),
        "left_anti"
    )
    display(df_datas_faltantes)
    #verifica se teria alguma date que estaria faltando entre o menor e o maior valor

run_ember_eda(df_ember_intensity_bronze, "Intensidade de Carbono")
run_ember_eda(df_ember_demand_bronze, "Demanda de Eletricidade")